# Modelling

First model to predict the number of rings. We compare a linear regression (baseline) with a random forest, and track the runs with MLflow (`mlflow ui --backend-store-uri sqlite:///mlflow.db` to see them).

# Imports

In [1]:
import mlflow
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Data

We drop the 2 rows with `Height == 0` and split 80/20.

In [2]:
df = pd.read_csv("../data/abalone.csv")
df = df[df["Height"] > 0]

X = df.drop(columns="Rings")
y = df["Rings"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Models

`Sex` is one-hot encoded, the other columns are already numeric.

In [3]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("abalone")

models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(n_estimators=100, random_state=42),
}

for name, model in models.items():
    pipeline = Pipeline(
        [
            (
                "encode",
                ColumnTransformer([("sex", OneHotEncoder(), ["Sex"])], remainder="passthrough"),
            ),
            ("model", model),
        ]
    )
    with mlflow.start_run(run_name=name):
        pipeline.fit(X_train, y_train)
        pred = pipeline.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        mae = mean_absolute_error(y_test, pred)
        r2 = r2_score(y_test, pred)
        mlflow.log_param("model", name)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
        print(f"{name}: rmse={rmse:.2f} mae={mae:.2f} r2={r2:.2f}")

2026/10/22 10:40:15 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/22 10:40:15 INFO mlflow.store.db.utils: Updating database tables


2026/10/22 10:40:15 INFO mlflow.tracking.fluent: Experiment with name 'abalone' does not exist. Creating a new experiment.


linear_regression: rmse=2.22 mae=1.59 r2=0.57


random_forest: rmse=2.24 mae=1.58 r2=0.56


# Conclusions

Linear regression: RMSE 2.22, MAE 1.59, R2 0.57. Random forest: RMSE 2.24, MAE 1.58, R2 0.56.

Both are equivalent, so we keep the **linear regression**: it is simpler and easier to explain. We will use it in the next steps (a pipeline with the one-hot encoding of `Sex` + the model).